# 01. Limpieza de DETER (contexto para el KPI)

**Propósito.** Extraer y limpiar las alertas de deforestación DETER-Amazônia (INPE). En el Hito 3 DETER **ya no es variable objetivo ni predictora**: se conserva solo como contexto para un KPI (alertas y hectáreas por municipio y semana).

**Entrada:** servicio WFS de TerraBrasilis (`deter-amz:deter_amz`), periodo 2022-01-01 a 2026-11-01. Si existen, `DETER/deter_limpio.csv` y `DETER/deter_limpio_merge.csv` del Hito 2 (se respaldan antes de sobrescribir).

**Salidas** (carpeta `DETER/`): `deter_limpio.csv`, `control_snapshot_deter.csv`, y los respaldos `deter_limpio_hito2.csv` y `deter_limpio_merge_hito2.csv`.

**Qué cambió respecto al Hito 2**
- Extracción nueva y completa (2022 a 2026-11-01), hecha por año, con reintentos. Reemplaza el snapshot congelado del Hito 2 (ver celda de texto siguiente).
- Se eliminó todo el cálculo de puntaje y nivel de riesgo de las alertas (ya no es necesario: DETER no es target).
- Se agrega la columna `semana` (lunes de la semana) con la misma función en todos los notebooks.
- Se agrega el control del snapshot (cuánto cambió INPE entre la versión del Hito 2 y la nueva).

In [12]:
from pathlib import Path
import unicodedata

import numpy as np
import pandas as pd

# ---------------------------------------------------------------------------
# 1. Entorno: Google Colab (Drive) o ejecucion local
# ---------------------------------------------------------------------------
USAR_DRIVE = True   # True en Colab; False para trabajar con la carpeta local del repositorio
BASE_DRIVE = Path("/content/drive/MyDrive/BIODIVERSITY_GUARD")

if USAR_DRIVE:
    try:
        from google.colab import drive
        drive.mount("/content/drive")
        BASE = BASE_DRIVE
    except ImportError:
        print("No se detecto Google Colab: se usa la raiz local del repositorio.")
        USAR_DRIVE = False

if not USAR_DRIVE:
    # la raiz es la primera carpeta (hacia arriba desde la actual) que contiene "datos"
    _aqui = Path.cwd().resolve()
    BASE = next((p for p in [_aqui, *_aqui.parents] if (p / "datos").is_dir()), None)
    if BASE is None:
        raise FileNotFoundError("No se encontro la carpeta 'datos' subiendo desde " + str(_aqui))

if not BASE.exists():
    raise FileNotFoundError(f"No existe BASE = {BASE}. Verifica la carpeta BIODIVERSITY_GUARD en Drive.")

# ---------------------------------------------------------------------------
# 2. Rutas (todas salen de BASE)
# ---------------------------------------------------------------------------
DIR_PREP = BASE / "datos" / "datasets_preprocesamiento"
DIR_BDQ = DIR_PREP / "BDQueimadas"
DIR_DETER = DIR_PREP / "DETER"
DIR_NASA = DIR_PREP / "NasaPower"
DIR_SISAM = DIR_PREP / "SISAM"
DIR_EDA = BASE / "datos" / "datasets_eda"

# ---------------------------------------------------------------------------
# 3. Constantes comunes (identicas en los 7 notebooks)
# ---------------------------------------------------------------------------
FECHA_INICIO_DATOS = "2022-01-01"
FECHA_FIN_EXTRACCION = "2026-11-01"      # DETER y BDQueimadas
PRIMERA_SEMANA = "2022-01-03"             # primer lunes con semana completa (2022-01-01 es sabado)
ULTIMA_SEMANA = "2026-10-19"              # lunes de la ultima semana completa (termina domingo 2026-10-25)
SATELITE_REFERENCIA = "AQUA_M-T"
ESTADOS_AMAZONIA_LEGAL = ["ACRE", "AMAPÁ", "AMAZONAS", "MARANHÃO", "MATO GROSSO", "PARÁ", "RONDÔNIA", "RORAIMA", "TOCANTINS"]
MAPA_UF = {"ACRE": "AC", "AMAPÁ": "AP", "AMAZONAS": "AM", "MARANHÃO": "MA", "MATO GROSSO": "MT", "PARÁ": "PA", "RONDÔNIA": "RO", "RORAIMA": "RR", "TOCANTINS": "TO"}
CORRECCIONES_MUNICIPIO = {"ELDORADO DOS CARAJAS": "ELDORADO DO CARAJAS", "SANTA ISABEL DO PARA": "SANTA IZABEL DO PARA"}
FECHA_CORTE_CENTROIDES = "2024-12-31"     # los centroides se fijan con focos hasta esta fecha

# ---------------------------------------------------------------------------
# 4. Funciones comunes
# ---------------------------------------------------------------------------
def normalizar_texto(texto):
    # mayusculas, sin tildes ni espacios sobrantes
    if pd.isna(texto):
        return texto
    texto = str(texto).upper().strip()
    return unicodedata.normalize("NFKD", texto).encode("ASCII", "ignore").decode("utf-8")


def semana_lunes(fecha):
    # lunes de la semana (lunes a domingo) de cada fecha; recibe una Serie de fechas
    f = pd.to_datetime(fecha).dt.normalize()
    return f - pd.to_timedelta(f.dt.day_of_week, unit="D")


SEMANAS = pd.date_range(PRIMERA_SEMANA, ULTIMA_SEMANA, freq="W-MON")   # calendario de semanas completas
assert len(SEMANAS) == 251, f"El calendario deberia tener 251 semanas y tiene {len(SEMANAS)}"
assert (SEMANAS.day_of_week == 0).all(), "Todas las semanas deben empezar en lunes"


def verificar_entradas(archivos):
    # imprime los archivos de entrada con su tamano y falla si falta alguno
    faltan = []
    for a in map(Path, archivos):
        if a.exists():
            print(f"  OK  {a.relative_to(BASE)}  ({a.stat().st_size / 1e6:,.1f} MB)")
        else:
            faltan.append(a)
    if faltan:
        lista = "\n".join(f"  - {a}" for a in faltan)
        raise FileNotFoundError("Faltan archivos de entrada (copialos a esas carpetas y vuelve a ejecutar):\n" + lista)


print("BASE:", BASE)
print("Calendario:", SEMANAS[0].date(), "a", SEMANAS[-1].date(), f"({len(SEMANAS)} semanas)")

# shapely no viene instalado por defecto en todos los entornos; se instala solo si falta
import importlib.util, subprocess, sys
if importlib.util.find_spec("shapely") is None:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "shapely"])

DIR_DETER.mkdir(parents=True, exist_ok=True)
print("\nEntrada: servicio WFS (requiere internet). Salida:", DIR_DETER.relative_to(BASE))
print("Archivos existentes en DETER/:", sorted(p.name for p in DIR_DETER.glob("*.csv")))

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
BASE: /content/drive/MyDrive/BIODIVERSITY_GUARD
Calendario: 2022-01-03 a 2026-10-19 (251 semanas)

Entrada: servicio WFS (requiere internet). Salida: datos/datasets_preprocesamiento/DETER
Archivos existentes en DETER/: ['control_snapshot_deter.csv', 'deter_limpio.csv', 'deter_limpio_hito2.csv', 'deter_limpio_merge.csv', 'deter_limpio_merge_hito2.csv']


## Respaldo de los archivos del Hito 2

Esta ejecución hace una extracción nueva y completa, así que sobrescribe `deter_limpio.csv`. Antes de hacerlo se copian los archivos del Hito 2 a `*_hito2.csv` (solo si el respaldo aún no existe, para no pisarlo en una segunda ejecución). `deter_limpio_merge.csv` ya no se usa en el Hito 3, pero también se respalda.

In [13]:
import shutil

for original, respaldo in [("deter_limpio.csv", "deter_limpio_hito2.csv"),
                           ("deter_limpio_merge.csv", "deter_limpio_merge_hito2.csv")]:
    ruta_o, ruta_r = DIR_DETER / original, DIR_DETER / respaldo
    if ruta_o.exists() and not ruta_r.exists():
        shutil.copy2(ruta_o, ruta_r)
        print(f"Respaldo creado: {respaldo}")
    elif ruta_r.exists():
        print(f"Ya existia el respaldo {respaldo}: no se toca")
    else:
        print(f"No existe {original}: nada que respaldar")

Ya existia el respaldo deter_limpio_hito2.csv: no se toca
Ya existia el respaldo deter_limpio_merge_hito2.csv: no se toca


## Extracción nueva y completa (2022-01-01 a 2026-11-01)

**Decisión:** el snapshot del Hito 2 queda reemplazado por una extracción nueva completa, no anexada. INPE revisa y reclasifica alertas históricas, por lo que la extracción se rehace entera y se mide la diferencia en la sección de control del snapshot.

La extracción se hace **por año** (un `CQL_FILTER` por año) con 3 reintentos y espera creciente. El servidor WFS parece devolver como máximo 50,000 filas por consulta (la extracción del Hito 2 de tres años dio exactamente 50,000 alertas). Por eso, si un año devuelve 50,000 filas o más, se vuelve a pedir **mes a mes** y se verifica que ningún mes llegue al tope.

In [14]:
import io
import time

import requests

URL_WFS = "https://terrabrasilis.dpi.inpe.br/geoserver/wfs"
ANIOS = list(range(pd.Timestamp(FECHA_INICIO_DATOS).year, pd.Timestamp(FECHA_FIN_EXTRACCION).year + 1))
TOPE_SERVIDOR = 50000      # tope sospechado de filas por consulta WFS
REINTENTOS = 3
ESPERA_BASE_S = 5          # espera creciente: 5, 10, 15 segundos
TIMEOUT_S = 300


def pedir_wfs(ini, fin):
    # una consulta WFS (CSV) para el rango de view_date [ini, fin], con reintentos
    params = {
        "service": "WFS", "version": "2.0.0", "request": "GetFeature",
        "typeNames": "deter-amz:deter_amz", "outputFormat": "csv",
        "CQL_FILTER": f"view_date BETWEEN '{ini}' AND '{fin}'",
    }
    for intento in range(1, REINTENTOS + 1):
        try:
            r = requests.get(URL_WFS, params=params, timeout=TIMEOUT_S)
            r.raise_for_status()
            return pd.read_csv(io.StringIO(r.text))
        except (requests.RequestException, pd.errors.ParserError, pd.errors.EmptyDataError) as e:
            print(f"  intento {intento}/{REINTENTOS} fallo para {ini} a {fin}: {e}")
            if intento == REINTENTOS:
                raise
            time.sleep(ESPERA_BASE_S * intento)


def rangos_mensuales(ini, fin):
    ini, fin = pd.Timestamp(ini), pd.Timestamp(fin)
    for p in pd.period_range(ini, fin, freq="M"):
        yield max(p.start_time, ini).strftime("%Y-%m-%d"), min(p.end_time.normalize(), fin).strftime("%Y-%m-%d")


def descargar_anio(anio):
    ini = max(f"{anio}-01-01", FECHA_INICIO_DATOS)
    fin = min(f"{anio}-12-31", FECHA_FIN_EXTRACCION)
    d = pedir_wfs(ini, fin)
    if len(d) >= TOPE_SERVIDOR:
        print(f"  AVISO {anio}: {len(d):,} filas (posible tope del servidor). Se repite mes a mes.")
        partes = []
        for m_ini, m_fin in rangos_mensuales(ini, fin):
            p = pedir_wfs(m_ini, m_fin)
            assert len(p) < TOPE_SERVIDOR, f"El mes {m_ini} a {m_fin} llego al tope del servidor ({len(p)} filas): hay que dividir por semanas"
            partes.append(p)
        d = pd.concat(partes, ignore_index=True)
    return d


partes_anio = {}
for a in ANIOS:
    partes_anio[a] = descargar_anio(a)
    print(f"{a}: {len(partes_anio[a]):,} filas")

df = pd.concat(partes_anio.values(), ignore_index=True)
print("\nTotal extraido:", df.shape)
print("Columnas:", df.columns.tolist())

  AVISO 2022: 50,000 filas (posible tope del servidor). Se repite mes a mes.
2022: 53,088 filas
2023: 40,186 filas
  AVISO 2024: 50,000 filas (posible tope del servidor). Se repite mes a mes.
2024: 81,229 filas
2025: 23,209 filas
2026: 24,117 filas

Total extraido: (221829, 16)
Columnas: ['FID', 'gid', 'classname', 'quadrant', 'path_row', 'view_date', 'sensor', 'satellite', 'areauckm', 'uc', 'areamunkm', 'municipality', 'mun_geocod', 'uf', 'publish_month', 'geom']


### Hallazgos de esta etapa

**Qué se observa**
- La extracción nueva trae **221,829 filas** crudas para 2022 a 2026.
- Al pedir 2022 y 2024 de una sola vez, el servicio devolvió exactamente **50,000 filas**. Se repitió mes a mes y salieron **53,088** (2022) y **81,229** (2024).
- El extracto del Hito 2 tenía **49,971 alertas** para 2022 a 2024, casi igual al tope de 50,000.

**Por qué se decidió así**
- El servicio WFS corta la respuesta en 50,000 filas. Por eso se extrae por año y, si un año llega al tope, por mes.
- El notebook avisa cuando un año llega a 50,000 filas, para no perder datos sin darse cuenta.

**Qué implica para el modelado**
- **Limitación:** el extracto del Hito 2 estaba **truncado por el tope del servicio**. Las cifras de DETER del Hito 2 no son comparables con las de esta versión.


## Limpieza inicial

Se eliminan columnas sin valor predictivo: `quadrant` (100 % vacía), `FID` (redundante con `gid`) y `path_row` (metadato de escena).

In [15]:
df = df.drop(columns=["quadrant", "FID", "path_row"], errors="ignore")

df["view_date"] = pd.to_datetime(df["view_date"], errors="coerce")
df["publish_month"] = pd.to_datetime(df["publish_month"], errors="coerce")
print("Fechas invalidas en view_date:", df["view_date"].isna().sum())
print("Fechas invalidas en publish_month:", df["publish_month"].isna().sum())

antes = len(df)
df = df.dropna(subset=["view_date"])
print(f"Filas eliminadas por fecha invalida: {antes - len(df)}")
print("Rango de fechas:", df["view_date"].min().date(), "a", df["view_date"].max().date())

Fechas invalidas en view_date: 0
Fechas invalidas en publish_month: 0
Filas eliminadas por fecha invalida: 0
Rango de fechas: 2022-01-01 a 2026-10-30


## Centroide del polígono y filtro de coordenadas

El centroide se calcula desde `geom` (WKT). El servicio WFS 2.0 entrega las coordenadas de EPSG:4326 en orden (latitud, longitud), por eso la coordenada `x` del centroide es la latitud. El filtro de Brasil (latitud entre -34 y 6, longitud entre -75 y -33) detecta si el orden estuviera invertido: se comprueba que se conserve casi todo.

In [16]:
from shapely import wkt


def extraer_centroide(geom_wkt):
    # devuelve (x, y) del centroide del poligono; NaN si la geometria no se puede leer
    try:
        c = wkt.loads(geom_wkt).centroid
        return c.x, c.y
    except Exception:
        return np.nan, np.nan


centroides = df["geom"].apply(extraer_centroide)
df["lat"] = [c[0] for c in centroides]
df["lon"] = [c[1] for c in centroides]
print("Geometrias que fallaron:", df["lat"].isna().sum())

antes = len(df)
df = df[df["lat"].between(-34, 6) & df["lon"].between(-75, -33)]
eliminadas = antes - len(df)
print(f"Filas eliminadas por coordenadas fuera de Brasil: {eliminadas}")
assert eliminadas / antes < 0.01, "Mas del 1 % fuera de Brasil: revisar el orden de ejes (lat, lon) del WKT"
print(df[["municipality", "uf", "lat", "lon"]].describe())

Geometrias que fallaron: 0
Filas eliminadas por coordenadas fuera de Brasil: 0
                 lat            lon
count  221829.000000  221829.000000
mean       -6.722641     -57.134180
std         3.820969       6.068715
min       -17.160785     -73.664507
25%        -9.503845     -61.221965
50%        -7.259148     -56.010013
75%        -3.597745     -52.474067
max         5.010199     -44.029219


In [17]:
# 1 km2 = 100 ha
df["area_ha"] = df["areamunkm"] * 100
antes = len(df)
df = df[df["area_ha"] > 0]
print(f"Filas eliminadas por area <= 0: {antes - len(df)}")

df["dentro_area_protegida"] = df["uc"].notna().astype(int)
print(f"Porcentaje dentro de area protegida: {df['dentro_area_protegida'].mean() * 100:.2f}%")
print("\nAlertas por clase:")
print(df["classname"].value_counts())

Filas eliminadas por area <= 0: 149
Porcentaje dentro de area protegida: 5.68%

Alertas por clase:
classname
DESMATAMENTO_CR         109136
CICATRIZ_DE_QUEIMADA     81218
DEGRADACAO               16282
CS_DESORDENADO            4090
MINERACAO                 3822
CS_GEOMETRICO             3654
DESMATAMENTO_VEG          3478
Name: count, dtype: int64


### Hallazgos de esta etapa

**Qué se observa**
- No se eliminó ninguna fila por fecha inválida (**0**), por geometría ilegible (**0**) ni por coordenadas fuera de Brasil (**0**).
- Solo se eliminaron **149 filas** con área menor o igual a 0. Quedan **221,680** alertas.
- Las clases más frecuentes son `DESMATAMENTO_CR` (**109,136**) y `CICATRIZ_DE_QUEIMADA` (**81,218**). El **5.68 %** de las alertas cae dentro de un área protegida.

**Por qué se decidió así**
- Una alerta sin área no aporta a la suma de hectáreas del KPI.
- **DETER es solo contexto para un KPI** (alertas y hectáreas por municipio y semana). No es variable objetivo ni predictora: el objetivo del Hito 3 son los focos de calor.


## Identificador único (`gid` NO es único)

En la exploración del Hito 2 se comprobó que `gid` se repite entre lotes históricos de INPE (sufijo `_hist`) y corresponde a alertas **distintas**, no a duplicados reales. Por eso no se usa como clave ni se deduplica por `gid`: se genera `id_alerta` propio y se verifica que no haya filas 100 % idénticas.

In [18]:
df = df.reset_index(drop=True)
df["id_alerta"] = df.index

print("Filas con gid repetido (informativo):", df.duplicated(subset=["gid"]).sum())
print("Filas 100% identicas (todas las columnas):", df.drop(columns=["id_alerta"]).duplicated().sum())
assert df["id_alerta"].is_unique

df["municipio_norm"] = df["municipality"].apply(normalizar_texto)
df["semana"] = semana_lunes(df["view_date"])
print("Municipios unicos (normalizados):", df["municipio_norm"].nunique())
print("UF:", sorted(df["uf"].dropna().unique()))

Filas con gid repetido (informativo): 103705
Filas 100% identicas (todas las columnas): 0
Municipios unicos (normalizados): 478
UF: ['AC', 'AM', 'AP', 'MA', 'MT', 'PA', 'RO', 'RR', 'TO']


In [19]:
columnas_finales = [
    "id_alerta", "gid", "classname", "view_date", "publish_month", "sensor", "satellite",
    "municipality", "mun_geocod", "uf", "lat", "lon", "area_ha", "areauckm",
    "dentro_area_protegida", "uc", "municipio_norm", "semana",
]
df_final = df[columnas_finales].sort_values("view_date").reset_index(drop=True)

df_final.to_csv(DIR_DETER / "deter_limpio.csv", index=False, encoding="utf-8")
print("Guardado:", (DIR_DETER / "deter_limpio.csv").relative_to(BASE))
print(f"Filas: {len(df_final):,} | Columnas: {df_final.shape[1]}")
print("Rango:", df_final["view_date"].min().date(), "a", df_final["view_date"].max().date())
print("Alertas por anio:\n", df_final["view_date"].dt.year.value_counts().sort_index())

Guardado: datos/datasets_preprocesamiento/DETER/deter_limpio.csv
Filas: 221,680 | Columnas: 18
Rango: 2022-01-01 a 2026-10-30
Alertas por anio:
 view_date
2022    53069
2023    40161
2024    81159
2025    23194
2026    24097
Name: count, dtype: int64


## Control del snapshot (Hito 2 contra extracción nueva)

Compara, para 2022 a 2024, la versión anterior (`deter_limpio_hito2.csv`) con la nueva: número de alertas, hectáreas totales y alertas por clase y por año. Es un dato para el informe (cuánto revisó INPE sus alertas históricas). Una diferencia grande en el número de alertas también puede indicar que la extracción anterior estaba truncada por el tope del servidor.

In [20]:
ruta_h2 = DIR_DETER / "deter_limpio_hito2.csv"

if not ruta_h2.exists():
    print("No existe deter_limpio_hito2.csv: no hay snapshot anterior que comparar.")
else:
    viejo = pd.read_csv(ruta_h2, usecols=["classname", "view_date", "area_ha"], parse_dates=["view_date"])
    nuevo = df_final[["classname", "view_date", "area_ha"]]
    f_ini, f_fin = pd.Timestamp("2022-01-01"), pd.Timestamp("2024-12-31")
    viejo = viejo[viejo["view_date"].between(f_ini, f_fin)].copy()
    nuevo = nuevo[nuevo["view_date"].between(f_ini, f_fin)].copy()

    def resumen(d, tipo):
        d = d.copy()
        d["_t"] = "total"
        d["_a"] = d["view_date"].dt.year.astype(str)
        clave = {"total": "_t", "anio": "_a", "clase": "classname"}[tipo]
        g = d.groupby(clave).agg(alertas=("area_ha", "size"), hectareas=("area_ha", "sum"))
        g.index = pd.MultiIndex.from_product([[tipo], g.index.astype(str)])
        return g

    tablas = []
    for tipo in ["total", "anio", "clase"]:
        t = resumen(viejo, tipo).join(resumen(nuevo, tipo), lsuffix="_hito2", rsuffix="_nuevo", how="outer").fillna(0)
        t["dif_alertas"] = t["alertas_nuevo"] - t["alertas_hito2"]
        t["dif_alertas_pct"] = (t["dif_alertas"] / t["alertas_hito2"].replace(0, np.nan) * 100).round(2)
        t["dif_hectareas"] = t["hectareas_nuevo"] - t["hectareas_hito2"]
        tablas.append(t)
    control = pd.concat(tablas)
    control.index.names = ["nivel", "clave"]
    control = control.reset_index().round(2)

    control.to_csv(DIR_DETER / "control_snapshot_deter.csv", index=False, encoding="utf-8")
    print("Guardado:", (DIR_DETER / "control_snapshot_deter.csv").relative_to(BASE))
    print(control.to_string(index=False))

Guardado: datos/datasets_preprocesamiento/DETER/control_snapshot_deter.csv
nivel                clave  alertas_hito2  hectareas_hito2  alertas_nuevo  hectareas_nuevo  dif_alertas  dif_alertas_pct  dif_hectareas
total                total          49971       3334745.18         174389      10964761.67       124418           248.98     7630016.49
 anio                 2022          14593        800800.16          53069       2829215.17        38476           263.66     2028415.01
 anio                 2023          11012        572109.93          40161       2051221.66        29149           264.70     1479111.73
 anio                 2024          24366       1961835.09          81159       6084324.84        56793           233.08     4122489.75
clase CICATRIZ_DE_QUEIMADA          21041       2227842.90          69764       7054943.98        48723           231.56     4827101.08
clase       CS_DESORDENADO            721        179410.47           2655        647836.84         1934      

### Hallazgos de esta etapa

**Qué se observa**
- Para 2022 a 2024, el Hito 2 tenía **49,971 alertas** y la extracción nueva tiene **174,389** (**+248.98 %**). Las hectáreas pasan de **3,334,745.18** a **10,964,761.67**.
- Todas las clases aumentan entre **231.56 %** y **272.01 %**.

| Corte | Alertas Hito 2 | Alertas nuevo | Diferencia (%) |
|---|---|---|---|
| 2022 | 14,593 | 53,069 | 263.66 |
| 2023 | 11,012 | 40,161 | 264.70 |
| 2024 | 24,366 | 81,159 | 233.08 |
| `CICATRIZ_DE_QUEIMADA` | 21,041 | 69,764 | 231.56 |
| `DESMATAMENTO_CR` | 23,054 | 83,824 | 263.60 |
| `DEGRADACAO` | 3,215 | 11,149 | 246.78 |
| `CS_DESORDENADO` | 721 | 2,655 | 268.24 |
| `CS_GEOMETRICO` | 609 | 2,088 | 242.86 |
| `DESMATAMENTO_VEG` | 688 | 2,517 | 265.84 |
| `MINERACAO` | 643 | 2,392 | 272.01 |

**Por qué se decidió así**
- El aumento es parecido en todos los años y clases. Eso apunta a un corte del volumen (el tope de 50,000) y no a cambios de clasificación de INPE.
- Por eso el snapshot se reemplaza completo y la comparación queda guardada en `control_snapshot_deter.csv`.

**Qué implica para el modelado**
- **Limitación:** las hectáreas DETER del Hito 2 quedan reemplazadas y no se deben comparar con las nuevas.


## Diccionario de variables: `deter_limpio.csv`

Alertas DETER-Amazônia (INPE), 2022-01-01 a 2026-11-01. Una fila por alerta.

| Variable | Descripción |
|---|---|
| `id_alerta` | Identificador único propio (reemplaza a `gid`, que no es único). |
| `gid` | Identificador original de INPE. Solo referencia, no usar como clave. |
| `classname` | Tipo de alteración detectada (por ejemplo `DESMATAMENTO_CR`, `CICATRIZ_DE_QUEIMADA`, `DEGRADACAO`, `MINERACAO`). |
| `view_date` | Fecha de detección satelital. |
| `publish_month` | Mes de publicación oficial por INPE. |
| `sensor`, `satellite` | Sensor y satélite de la detección. |
| `municipality`, `mun_geocod`, `uf` | Municipio, código IBGE y estado (sigla). |
| `lat`, `lon` | Centroide del polígono de la alerta. |
| `area_ha` | Área de la alerta en hectáreas (`areamunkm` por 100). |
| `areauckm` | Área dentro de unidad de conservación (km²). |
| `dentro_area_protegida` | 1 si la alerta tiene unidad de conservación (`uc`), 0 si no. |
| `uc` | Nombre de la unidad de conservación. |
| `municipio_norm` | Nombre del municipio en mayúsculas y sin tildes (clave de cruce junto con `uf`). |
| `semana` | Lunes de la semana (lunes a domingo) de `view_date`. |

No se calcula ningún puntaje ni nivel de riesgo: DETER solo aporta contexto para el KPI.